In [82]:
# ==========================================================
# TRANSFORMER PARA SERIES DE TIEMPO
# - Input (window_size, 16)
# - Dense(d_model)
# - Positional Encoding
# - Transformer Encoder x N
# - Global Average Pooling
# - Dense
# - Predicción
#
# Validación:
# - Prequential Cross Validation
# - 3 folds por run
# - promedio por run
# - promedio final de promedios
#
# Salidas:
# - transformer-metrics.xlsx
# - transformer-resultados.xlsx
# - transformer-pasos.xlsx
# ==========================================================

import os
from time import time

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling1D,
    Input,
    Layer,
    LayerNormalization,
    MultiHeadAttention,
)
from tensorflow.keras.metrics import MeanAbsoluteError, MeanAbsolutePercentageError
from tensorflow.keras.optimizers import Adam



In [83]:
import tensorflow as tf

# Verificar si hay GPUs disponibles
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"¡GPU detectada! {len(gpus)} dispositivo(s) GPU disponibles:")
    for gpu in gpus:
        print(f"  - {gpu}")
    # Puedes verificar que una operación se ejecuta en la GPU
    with tf.device('/GPU:0'):
        a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
        b = tf.constant([[1.0, 1.0], [0.0, 1.0]])
        c = tf.matmul(a, b)
    print("\nOperación de ejemplo ejecutada en GPU:\n", c.numpy())
else:
    print("No se detectaron GPUs. Asegúrate de que el entorno de ejecución esté configurado para usar GPU.")


¡GPU detectada! 1 dispositivo(s) GPU disponibles:
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

Operación de ejemplo ejecutada en GPU:
 [[1. 3.]
 [3. 7.]]


In [84]:
# ==========================================================
# 1. CONFIGURACIÓN
# ==========================================================

RUTA_CSV = r"C:\Users\Ximena\DLPronostico\data.csv"

#-----------------------------------------------------------
#tuning 
#----------------------------------------------------------

WINDOW_SIZE = 10
FOLD_SIZE = 1512
N_FOLDS = 3
EPOCHS = 100
BATCH_SIZE = 128
D_MODEL = 64
NUM_ENCODER_BLOCKS = 2

#-------------------------------

#BASE_DIR = os.path.dirname(os.path.abspath(__file__))
BASE_DIR = os.getcwd()
print(f"Directorio base configurado en: {BASE_DIR}")

# TRAINING_VERBOSE = 2
TRAINING_VERBOSE = 1


def log(message=""):
    print(message, flush=True)



Directorio base configurado en: C:\Users\Ximena\DLPronostico\TRANSFORMER


In [85]:
# ==========================================================
# 2. TRATAMIENTO DE DATOS
# ==========================================================

def preparar_datos(ruta_csv):
    df = pd.read_csv(ruta_csv, parse_dates=["beginTimeSeconds"])
    df = df.sort_values("beginTimeSeconds")
    df = df.set_index("beginTimeSeconds")

    metricas = [
        "Free Memory",
        "Used Memory",
        "Free Disk",
        "Used Disk",
        "Disk read/s",
        "Disk write/s",
        "NetBytes In",
        "NetBytes Out",
        "NetPackets In",
        "NetPackets Out",
        "Rx packets",
        "Tx packets",
        "CPU percent",
        "Memory Used percent",
        "Disk Used percent",
        "Uptime",
    ]

    faltantes = [col for col in metricas if col not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas en el CSV: {faltantes}")

    df = df[metricas].copy()
    df = df.ffill().bfill()

    return df


def seleccionar_metricas(df):
    log("\nSeleccione las métricas objetivo a analizar:\n")
    columnas = list(df.columns)

    for i, col in enumerate(columnas, 1):
        log(f"{i}. {col}")

    seleccion = input("\nIngrese números o nombres separados por coma: ").strip()

    metricas = []
    for x in seleccion.split(","):
        x = x.strip()
        if not x:
            continue
        if x.isdigit():
            idx = int(x)
            if 1 <= idx <= len(columnas):
                metricas.append(columnas[idx - 1])
        elif x in columnas:
            metricas.append(x)

    metricas = list(dict.fromkeys(metricas))

    if not metricas:
        log("No se seleccionó ninguna métrica. Se usará 'Free Memory'.")
        metricas = ["Free Memory"]

    log(f"\nMétricas seleccionadas: {metricas}")
    return metricas


def seleccionar_numero_runs():
    log("\nSeleccione el número de runs:")
    log("1. 10")
    log("2. 30")
    log("3. 50")

    while True:
        seleccion = input("Ingrese 1, 2, 3 o 10, 30, 50: ").strip()

        if seleccion in ("1", "10"):
            return 10
        if seleccion in ("2", "30"):
            return 30
        if seleccion in ("3", "50"):
            return 50

        log("Entrada no válida. Solo se permite 10, 30 o 50.")


def seleccionar_modo():
    log("\nSeleccione el modo de predicción:")
    log("1. One-step")
    log("2. Multi-step")

    modo = input("Ingrese 1 o 2: ").strip()

    if modo == "2":
        try:
            n_steps = int(input("Ingrese el número de pasos para multi-step: ").strip())
            if n_steps < 1:
                n_steps = 5
        except Exception:
            n_steps = 5
        return "multi-step", n_steps

    return "one-step", None

#-----------------------------------------------------------
# GENERA LOS FOLDS PARA LA VALIDACIÓN PREQUENTIAL
#------------------------------------------------------------

def generar_folds(total_len, fold_size=1512, n_folds=3):
    folds = []

    for i in range(n_folds):
        train_end = fold_size * (i + 1)
        test_start = train_end
        test_end = min(train_end + fold_size, total_len)

        if train_end > total_len or test_start >= total_len:
            break

        folds.append((0, train_end, test_start, test_end))

    return folds

#-----------------------------------------------------------
# CREA LAS VENTANAS DE SECUENCIAS PARA EL MODELO TRANSFORMER
#------------------------------------------------------------
def crear_ventanas(X_data, y_data, window_size=10):
    X, y = [], []

    for i in range(len(X_data) - window_size):
        X.append(X_data[i:i + window_size])
        y.append(y_data[i + window_size])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)



In [86]:
# ==========================================================
# 3. MODELO TRANSFORMER
# ==========================================================

class PositionalEncoding(Layer):

    """
    Agrega información de posición temporal a cada paso de la secuencia.
    Esto permite que el Transformer distinga el orden de los datos,
    ya que la atención por sí sola no conoce la posición de cada elemento.
    """

    def __init__(self, max_len=5000, d_model=64, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model

         # pos: posiciones temporales [0, 1, 2, ..., max_len-1]
        pos = np.arange(max_len)[:, np.newaxis]

         # pos: posiciones temporales [0, 1, 2, ..., max_len-1]
        i = np.arange(d_model)[np.newaxis, :]

         # pos: posiciones temporales [0, 1, 2, ..., max_len-1]
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
        angle_rads = pos * angle_rates

        # matriz de codificación posicional
        pe = np.zeros((max_len, d_model))
        pe[:, 0::2] = np.sin(angle_rads[:, 0::2])
        pe[:, 1::2] = np.cos(angle_rads[:, 1::2])

        self.positional_encoding = tf.constant(pe[np.newaxis, ...], dtype=tf.float32)

    def call(self, inputs):

        # matriz de codificación posicional con la misma forma que la entrada
        seq_len = tf.shape(inputs)[1]

        # se suma la codificación posicional a la representación embebida
        return inputs + self.positional_encoding[:, :seq_len, :]


def rmse_tf(y_true, y_pred):

    """
    Función de pérdida basada en RMSE.
    """
    return tf.math.sqrt(tf.math.reduce_mean(tf.math.square(y_pred - y_true)))

    #se define el bloque encoder del diagrama
def transformer_encoder(x, head_size=16, num_heads=4, ff_dim=64, dropout=0.1):

    """
    Bloque Transformer Encoder.

    Estructura:
    1. Multi-Head Self-Attention
    2. Residual Connection + Layer Normalization
    3. Feed Forward Network
    4. Residual Connection + Layer Normalization
    """

    # ------------------------------------------------------
    # 1) Multi-Head Self-Attention
    # Cada paso de la ventana atiende a los demás pasos
    # para capturar dependencias temporales.
    # ------------------------------------------------------
    attn = MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(x, x)

    # ------------------------------------------------------
    # 2) Residual Connection + Layer Normalization
    # Se suma la entrada original del bloque con la salida
    # de atención para estabilizar el entrenamiento.
    # ------------------------------------------------------

    x = LayerNormalization(epsilon=1e-6)(x + Dropout(dropout)(attn))

    # ------------------------------------------------------
    # 3) Feed Forward Network
    # Red neuronal completamente conectada para capturar
    # relaciones no lineales.
    # ------------------------------------------------------
    ff = Dense(ff_dim, activation="relu")(x)
    ff = Dropout(dropout)(ff)
    ff = Dense(int(x.shape[-1]))(ff)

    # ------------------------------------------------------
    # 4) Residual Connection + Layer Normalization
    # Se suma la entrada original del bloque con la salida
    # de la red feed-forward para estabilizar el entrenamiento.
    # ------------------------------------------------------
    return LayerNormalization(epsilon=1e-6)(x + ff)


def crear_modelo_transformer(input_shape, d_model=64, num_blocks=2):

    """
    Construye el modelo Transformer para predicción de series temporales.

    Arquitectura general:
    Input (window_size, 16)
        -> Dense(d_model)              : embedding/proyección de features
        -> Positional Encoding         : agrega orden temporal
        -> Transformer Encoder x N     : atención + feed forward
        -> Global Average Pooling      : resume la ventana completa
        -> Dense                       : capa intermedia final
        -> Predicción                  : y(t+1)
    """

    # Entrada multivariada:
    # input_shape = (window_size, 16)

    inputs = Input(shape=input_shape)

    # ------------------------------------------------------
    # Embedding / proyección inicial
    # Convierte las 16 features a un espacio de dimensión d_model.
    # ------------------------------------------------------

    x = Dense(d_model)(inputs)

    # ------------------------------------------------------
    # Positional Encoding
    # Se añade información de posición dentro de la ventana temporal.
    # ------------------------------------------------------

    x = PositionalEncoding(max_len=input_shape[0], d_model=d_model)(x)

    # ------------------------------------------------------
    # Transformer Encoder x N
    # Se apilan varios bloques encoder para aprender patrones
    # temporales más complejos.
    # ------------------------------------------------------

    for _ in range(num_blocks):
        x = transformer_encoder(x, head_size=16, num_heads=4, ff_dim=64, dropout=0.1)

    
    # ------------------------------------------------------
    # Global Average Pooling
    # Resume toda la secuencia en un solo vector.
    # ------------------------------------------------------

    x = GlobalAveragePooling1D()(x)

    # ------------------------------------------------------
    # Capa densa final
    # Refina la representación antes de la salida.
    # ------------------------------------------------------

    x = Dense(32, activation="relu")(x)
    x = Dropout(0.1)(x)

    # ------------------------------------------------------
    # Capa de salida
    # Genera la predicción final.
    # ------------------------------------------------------
    outputs = Dense(1, activation="linear")(x)

    # ------------------------------------------------------
    # Construcción y compilación del modelo
    # ------------------------------------------------------
    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss=rmse_tf,
        metrics=[MeanAbsoluteError(), MeanAbsolutePercentageError()],
    )

    return model



In [87]:
# ==========================================================
# 4. FORECAST RECURSIVO MULTIVARIADO
# ==========================================================

def construir_siguiente_fila(ultima_fila, pred_target_scaled, target_idx):
    nueva_fila = ultima_fila.copy()
    nueva_fila[target_idx] = float(pred_target_scaled)
    return nueva_fila


def forecast_recursive_multivariado(
    model,
    X_hist_scaled,
    y_test_scaled,
    scaler_y,
    target_idx,
    window_size=10,
    horizon=None
):
    history = [fila.copy() for fila in X_hist_scaled]

    if horizon is None:
        horizon = len(y_test_scaled)

    horizon = min(horizon, len(y_test_scaled))

    y_pred_scaled = []
    y_true_scaled = []

    for paso in range(horizon):
        x_input = np.array(history[-window_size:], dtype=np.float32).reshape(
            1, window_size, X_hist_scaled.shape[1]
        )

        pred_scaled = float(model.predict(x_input, verbose=0)[0][0])

        y_pred_scaled.append(pred_scaled)
        y_true_scaled.append(float(y_test_scaled[paso][0]))

        nueva_fila = construir_siguiente_fila(
            ultima_fila=history[-1],
            pred_target_scaled=pred_scaled,
            target_idx=target_idx
        )
        history.append(nueva_fila)

    y_pred_scaled = np.array(y_pred_scaled, dtype=np.float32).reshape(-1, 1)
    y_true_scaled = np.array(y_true_scaled, dtype=np.float32).reshape(-1, 1)

    y_pred_inv = scaler_y.inverse_transform(y_pred_scaled)
    y_true_inv = scaler_y.inverse_transform(y_true_scaled)

    return y_true_inv, y_pred_inv


def calcular_metricas(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1, 1)
    y_pred = np.asarray(y_pred).reshape(-1, 1)

    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(np.mean(np.abs(y_true - y_pred)))

    denom = np.where(np.abs(y_true) < 1e-8, np.nan, np.abs(y_true))
    mape = float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100)

    return rmse, mae, mape


In [88]:

# ==========================================================
# 5. EXPORTACIÓN
# ==========================================================

def ordenar_por_metricas(df, metricas_orden):
    if df.empty or "Metric" not in df.columns:
        return df

    df = df.copy()
    metricas_presentes = [m for m in df["Metric"].dropna().astype(str).unique().tolist()]
    categorias = list(dict.fromkeys(metricas_orden + metricas_presentes))
    df["Metric"] = pd.Categorical(df["Metric"], categories=categorias, ordered=True)

    columnas_sort = [col for col in ["Metric", "Run", "Fold", "Step"] if col in df.columns]
    df = df.sort_values(columnas_sort).reset_index(drop=True)
    df["Metric"] = df["Metric"].astype(str)

    return df



In [89]:
def exportar_resultados(df_metrics, df_runs, df_final, df_pasos, metricas_orden):
    archivo_metrics = os.path.join(BASE_DIR, "transformer-metrics.xlsx")
    archivo_resultados = os.path.join(BASE_DIR, "transformer-resultados.xlsx")
    archivo_pasos = os.path.join(BASE_DIR, "transformer-pasos.xlsx")

    df_metrics = ordenar_por_metricas(df_metrics, metricas_orden)
    df_runs = ordenar_por_metricas(df_runs, metricas_orden)
    df_final = ordenar_por_metricas(df_final, metricas_orden)
    df_pasos = ordenar_por_metricas(df_pasos, metricas_orden)

    with pd.ExcelWriter(archivo_metrics, engine="openpyxl", mode="w") as writer:
        df_metrics.to_excel(writer, sheet_name="Runs_Folds", index=False)
        df_runs.to_excel(writer, sheet_name="Promedio_por_Run", index=False)

    with pd.ExcelWriter(archivo_resultados, engine="openpyxl", mode="w") as writer:
        df_final.to_excel(writer, sheet_name="Resumen_Final", index=False)

    with pd.ExcelWriter(archivo_pasos, engine="openpyxl", mode="w") as writer:
        df_pasos.to_excel(writer, sheet_name="Predicciones_Paso", index=False)

    log(f"\nArchivo generado: {archivo_metrics}")
    log(f"Archivo generado: {archivo_resultados}")
    log(f"Archivo generado: {archivo_pasos}")


In [90]:

# ==========================================================
# 6. EJECUCIÓN
# ==========================================================

def main():
    df = preparar_datos(RUTA_CSV)
    metricas_seleccionadas = seleccionar_metricas(df)
    n_runs = seleccionar_numero_runs()
    modo, n_steps = seleccionar_modo()

    folds = generar_folds(len(df), fold_size=FOLD_SIZE, n_folds=N_FOLDS)

    log("\nITERATIONS GENERATED:")
    for i, f in enumerate(folds):
        log(f"I{i} -> Train[0:{f[1]}] Test[{f[2]}:{f[3]}]")

    metrics_fold_rows = []
    resultados_runs_rows = []
    resultados_finales_rows = []
    pasos_rows = []

    X_full_global = df.values.astype(np.float32)

    for metrica in metricas_seleccionadas:
        log(f"\n{'=' * 80}")
        log(f"METRIC: {metrica}")
        log(f"{'=' * 80}")

        y_full = df[metrica].values.reshape(-1, 1).astype(np.float32)
        target_idx = df.columns.get_loc(metrica)

        promedios_runs_metrica = []

        for run in range(1, n_runs + 1):
            log(f"\nRUN {run}/{n_runs}")

            resultados_fold_metrica = []

            for fold_id, (train_start, train_end, test_start, test_end) in enumerate(folds):
                # log(f"  Fold F{fold_id}")
                log(
                    f"Iteration I{fold_id} | "
                    f"train={train_end - train_start} registros | "
                    f"test={test_end - test_start} registros"
                )

                X_train_full = X_full_global[train_start:train_end]
                y_train_full = y_full[train_start:train_end]
                y_test = y_full[test_start:test_end]

                if len(X_train_full) <= WINDOW_SIZE or len(y_test) == 0:
                    log("  Fold omitido por tamaño insuficiente.")
                    continue

                val_split = int(len(X_train_full) * 0.8)

                X_train = X_train_full[:val_split]
                X_val = X_train_full[val_split:]

                y_train = y_train_full[:val_split]
                y_val = y_train_full[val_split:]

                if len(X_train) <= WINDOW_SIZE or len(X_val) <= WINDOW_SIZE:
                    log("  Fold omitido por train/val insuficiente.")
                    continue

                scaler_X = MinMaxScaler()
                scaler_y = MinMaxScaler()

                X_train_scaled = scaler_X.fit_transform(X_train)
                X_val_scaled = scaler_X.transform(X_val)

                y_train_scaled = scaler_y.fit_transform(y_train)
                y_val_scaled = scaler_y.transform(y_val)
                y_test_scaled = scaler_y.transform(y_test)

                X_train_win, y_train_win = crear_ventanas(
                    X_train_scaled, y_train_scaled, window_size=WINDOW_SIZE
                )
                X_val_win, y_val_win = crear_ventanas(
                    X_val_scaled, y_val_scaled, window_size=WINDOW_SIZE
                )

                if len(X_train_win) == 0 or len(X_val_win) == 0:
                    log("  Fold omitido por ventanas insuficientes.")
                    continue

                tf.keras.backend.clear_session()

                #-------------------------------------------------------------------------
                # creacion del modelo
                # --------------------------------------------------------------------------
                
                model = crear_modelo_transformer(
                    input_shape=(X_train_win.shape[1], X_train_win.shape[2]),
                    d_model=D_MODEL,
                    num_blocks=NUM_ENCODER_BLOCKS
                )

                #---------------------------------------------------------------------
                # ENTRENAMIENTO Y PREDICCIÓN RECURSIVA
                #---------------------------------------------------------------------
                inicio_training = time()
                # log(
                #     "    Entrenando modelo "
                #     f"(train_windows={len(X_train_win)}, val_windows={len(X_val_win)}, epochs={EPOCHS})"
                # )
                log(
                    "    Iniciando entrenamiento | "
                    f"train_windows={len(X_train_win)} | "
                    f"val_windows={len(X_val_win)} | "
                    f"epochs={EPOCHS} | "
                    f"batch_size={BATCH_SIZE}"
                )


                history = model.fit(
                    X_train_win,
                    y_train_win,
                    validation_data=(X_val_win, y_val_win),
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    verbose=TRAINING_VERBOSE,
                    callbacks=[
                        tf.keras.callbacks.EarlyStopping(
                            monitor="val_loss",
                            patience=5,
                            restore_best_weights=True
                        )
                    ],
                )


                time_training_ms = (time() - inicio_training) * 1000
                epochs_entrenadas = len(history.history.get("loss", []))
                mejor_val_loss = history.history.get("val_loss", [])
                mejor_val_loss = min(mejor_val_loss) if mejor_val_loss else float("nan")

                # log(
                #     f"    Entrenamiento F{fold_id} completado | "
                #     f"epochs={epochs_entrenadas} | best_val_loss={mejor_val_loss:.6f}"
                # )
                log(
                    f"    Entrenamiento F{fold_id} completado | "
                    f"epochs={epochs_entrenadas} | "
                    f"best_val_loss={mejor_val_loss:.6f}"
                )

                X_hist_scaled = np.vstack([X_train_scaled, X_val_scaled]).astype(np.float32)

                horizon = n_steps if modo == "multi-step" else len(y_test_scaled)

                y_true_inv, y_pred_inv = forecast_recursive_multivariado(
                    model=model,
                    X_hist_scaled=X_hist_scaled,
                    y_test_scaled=y_test_scaled,
                    scaler_y=scaler_y,
                    target_idx=target_idx,
                    window_size=WINDOW_SIZE,
                    horizon=horizon
                )

                rmse, mae, mape = calcular_metricas(y_true_inv, y_pred_inv)

                # log(
                #     f"    RESULTADO F{fold_id} | "
                #     f"RMSE={rmse:.4f} | "
                #     f"MAE={mae:.4f} | "
                #     f"MAPE={mape:.4f} | "
                #     f"TIME_TRAINING={time_training_ms:.2f} ms"
                # )
                log(
                    f"    RESULTADO I{fold_id} | "
                    f"RMSE={rmse:.4f} | "
                    f"MAE={mae:.4f} | "
                    f"MAPE={mape:.4f} | "
                    f"TIME_TRAINING={time_training_ms:.2f} ms"
                )

                metrics_fold_rows.append({
                    "Metric": metrica,
                    "Run": run,
                    "Iteration": fold_id,
                    "RMSE": rmse,
                    "MAE": mae,
                    "MAPE": mape,
                    "Time_Training_ms": time_training_ms,
                    "Mode": modo
                })

                resultados_fold_metrica.append([rmse, mae, mape, time_training_ms])

                #for paso, (real, pred) in enumerate(zip(y_true_inv.flatten(), y_pred_inv.flatten()), start=1):
                 #   idx_tiempo = test_start + (paso - 1)
                  #  fecha = df.index[idx_tiempo] if idx_tiempo < len(df.index) else pd.NaT

                   # pasos_rows.append({
                    #    "Metric": metrica,
                    #    "Run": run,
                    #    "Iteration": fold_id,
                    #    "Step": paso,
                    #    "Timestamp": fecha,
                    #    "Real": float(real),
                    #    "Pred": float(pred),
                    #    "Abs_Error": abs(float(real) - float(pred)),
                    #    "Mode": modo
                   # })

            if resultados_fold_metrica:
                resultados_fold_metrica = np.array(resultados_fold_metrica, dtype=np.float32)
                promedio_run = resultados_fold_metrica.mean(axis=0)

                resultados_runs_rows.append({
                    "Metric": metrica,
                    "Run": run,
                    "RMSE_mean": float(promedio_run[0]),
                    "MAE_mean": float(promedio_run[1]),
                    "MAPE_mean": float(promedio_run[2]),
                    "Time_Training_mean_ms": float(promedio_run[3]),
                    "Mode": modo
                })

                promedios_runs_metrica.append(promedio_run)

                log(
                    f"PROMEDIO RUN {run} | "
                    f"RMSE={promedio_run[0]:.4f} | "
                    f"MAE={promedio_run[1]:.4f} | "
                    f"MAPE={promedio_run[2]:.4f} | "
                    f"TIME_TRAINING={promedio_run[3]:.2f} ms"
                )

        if promedios_runs_metrica:
            promedios_runs_metrica = np.array(promedios_runs_metrica, dtype=np.float32)
            promedio_final = promedios_runs_metrica.mean(axis=0)

            resultados_finales_rows.append({
                "Metric": metrica,
                "RMSE_final_mean": float(promedio_final[0]),
                "MAE_final_mean": float(promedio_final[1]),
                "MAPE_final_mean": float(promedio_final[2]),
                "Time_Training_final_mean_ms": float(promedio_final[3]),
                "Mode": modo
            })

            log(f"\nPROMEDIO FINAL - {metrica}")
            log(
                f"RMSE={promedio_final[0]:.4f} | "
                f"MAE={promedio_final[1]:.4f} | "
                f"MAPE={promedio_final[2]:.4f} | "
                f"TIME_TRAINING={promedio_final[3]:.2f} ms"
            )

    df_metrics = pd.DataFrame(metrics_fold_rows)
    df_runs = pd.DataFrame(resultados_runs_rows)
    df_final = pd.DataFrame(resultados_finales_rows)
    df_pasos = pd.DataFrame(pasos_rows)

    exportar_resultados(
        df_metrics=df_metrics,
        df_runs=df_runs,
        df_final=df_final,
        df_pasos=df_pasos,
        metricas_orden=metricas_seleccionadas
    )

    log(f"\n{'=' * 80}")
    log("PROCESO COMPLETADO")
    log(f"{'=' * 80}")


if __name__ == "__main__":
    main()
    #if __name__ == "__main__":
     #   main()
     #   print("PROCESO COMPLETADO")


Seleccione las métricas objetivo a analizar:

1. Free Memory
2. Used Memory
3. Free Disk
4. Used Disk
5. Disk read/s
6. Disk write/s
7. NetBytes In
8. NetBytes Out
9. NetPackets In
10. NetPackets Out
11. Rx packets
12. Tx packets
13. CPU percent
14. Memory Used percent
15. Disk Used percent
16. Uptime



Ingrese números o nombres separados por coma:  11



Métricas seleccionadas: ['Rx packets']

Seleccione el número de runs:
1. 10
2. 30
3. 50


Ingrese 1, 2, 3 o 10, 30, 50:  10



Seleccione el modo de predicción:
1. One-step
2. Multi-step


Ingrese 1 o 2:  2
Ingrese el número de pasos para multi-step:  5



ITERATIONS GENERATED:
I0 -> Train[0:1512] Test[1512:3024]
I1 -> Train[0:3024] Test[3024:4536]
I2 -> Train[0:4536] Test[4536:6048]

METRIC: Rx packets

RUN 1/10
Iteration I0 | train=1512 registros | test=1512 registros
    Iniciando entrenamiento | train_windows=1199 | val_windows=293 | epochs=100 | batch_size=128
Epoch 1/100
10/10 [==============================] - 1s 38ms/step - loss: 0.6364 - mean_absolute_error: 0.5296 - mean_absolute_percentage_error: 13911412.0000 - val_loss: 0.1600 - val_mean_absolute_error: 0.1497 - val_mean_absolute_percentage_error: 5601683.0000
Epoch 2/100
10/10 [==============================] - 0s 19ms/step - loss: 0.5062 - mean_absolute_error: 0.3850 - mean_absolute_percentage_error: 9236108.0000 - val_loss: 0.3428 - val_mean_absolute_error: 0.3300 - val_mean_absolute_percentage_error: 8749198.0000
Epoch 3/100
10/10 [==============================] - 0s 19ms/step - loss: 0.4146 - mean_absolute_error: 0.3275 - mean_absolute_percentage_error: 13209016.0000 